# WC 2026 Monte Carlo Simulation

Run `simulate_world_cup` 100 times and aggregate results.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wc_2026_simulator_function import simulate_world_cup

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

CSV  = '../data/wc_2026_match_probabilities_avg.csv'
N    = 100
TEMP = 0.5

In [ ]:
results = []
for i in range(N):
    r = simulate_world_cup(CSV, verbose=False, seed=i, temperature=TEMP)
    results.append(r)

df = pd.DataFrame(results)
print(f'Simulations run: {len(df)}')
df.head(10)

## Champion frequency

In [ ]:
champ_counts = df['champion'].value_counts()
print(champ_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Blues_d', len(champ_counts))
bars = ax.barh(champ_counts.index[::-1], champ_counts.values[::-1], color=colors)
for bar, val in zip(bars, champ_counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val}', va='center', fontsize=9)
ax.set_xlabel(f'Times champion (out of {N})')
ax.set_title(f'WC 2026 Champion Frequency — {N} simulations (temp={TEMP})', fontweight='bold')
plt.tight_layout()
plt.show()

## Podium finish rates (top 4)

In [ ]:
podium_cols = ['champion', 'runner_up', 'third', 'fourth']
all_teams = sorted(set(df[podium_cols].values.flatten()))

podium_df = pd.DataFrame(index=all_teams)
for col in podium_cols:
    podium_df[col] = df[col].value_counts().reindex(all_teams, fill_value=0)

podium_df['total_podium'] = podium_df[podium_cols].sum(axis=1)
podium_df = podium_df.sort_values('champion', ascending=False)
podium_df

In [ ]:
top_n = podium_df[podium_df['total_podium'] > 0].copy()

fig, ax = plt.subplots(figsize=(12, max(4, len(top_n) * 0.4)))
bottom = pd.Series(0, index=top_n.index)
palette = sns.color_palette('Set2', 4)

for col, color, label in zip(
    podium_cols,
    palette,
    ['Champion', 'Runner-up', '3rd Place', '4th Place']
):
    ax.barh(top_n.index, top_n[col], left=bottom, color=color, label=label)
    bottom += top_n[col]

ax.set_xlabel(f'Podium finishes (out of {N})')
ax.set_title(f'WC 2026 Podium Finishes — {N} simulations (temp={TEMP})', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()